# Fit Model on full Dev Set

- Use model config derrived for each of self/parent cohorts in exploration to fit a single model
- Extract coefs from each trained self/parent model

In [ ]:
import sys, os

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED
from src.residual_wrapper import get_df_path
from src.coef_visual import plot_heatmap
from src.utils import col_config_sensitivity
from src.residual_func import ResidualRegressor
import pandas as pd

In [ ]:
shared_dict = {
    "num_layers": 3,
    "max_iter": 50000,
    "seed": SEED,
    "linear_type": "ridge",
}
parent_config = {
    "impact_type": "parent",
    "quest_cols": col_config_sensitivity["parent_quest"],
    "agg_type": "max",
    "alpha2": 10.0,
    "alpha3": 5.0,
    "desc_seg_type": "pt_noshort",
    **shared_dict,
}
self_config = {
    "impact_type": "self",
    "quest_cols": col_config_sensitivity["self_quest"],
    "agg_type": "block",
    "alpha2": 40.0,
    "alpha3": 0.5,
    "desc_seg_type": "turns",
    **shared_dict,
}

In [ ]:
def build_model(config_dict, llama_model, data_dir):
    data_path = get_df_path(
        df_dir=data_dir,
        llama_model=llama_model,
        agg_type=config_dict["agg_type"],
        desc_seg_type=config_dict["desc_seg_type"],
        impact_type=config_dict["impact_type"],
    )
    model = ResidualRegressor(**config_dict, two_layer_base=None)
    train_df = pd.read_parquet(data_path)
    print(len(train_df))
    df_expand = model.expand_text(train_df)
    target_col = f"impact_score_{config_dict['impact_type']}"
    X_train = df_expand.drop(target_col, axis=1)
    y_train = df_expand[target_col]
    model.fit(X_train, y_train, shuffle=None, perm_dir=None)
    # Process coef df
    coef_df = model.get_coef_df()
    coef_df["stage_feat"] = coef_df["stage"] + "_" + coef_df["feature"]
    coef_df = coef_df.drop(columns=["stage", "feature"])
    # remove this so can combine self+parent
    replace_str = "p_dep" if config_dict["impact_type"] == "parent" else "s_dep"
    coef_df["stage_feat"] = coef_df["stage_feat"].str.replace(
        replace_str, "", regex=False
    )
    coef_df = coef_df.sort_values("stage_feat")
    coef_df = coef_df.set_index("stage_feat")
    return model, coef_df, train_df

In [ ]:
data_dir = BASE_PATH / "data"
llama_model = "Llama-3.3-70B-Instruct"
model_parent, coef_parent, df_parent = build_model(parent_config, llama_model, data_dir)
model_self, coef_self, df_self = build_model(self_config, llama_model, data_dir)

In [ ]:
coef_parent = coef_parent.rename(columns={"coef": "coef_parent"})
coef_self = coef_self.rename(columns={"coef": "coef_self"})
coef_combined = coef_parent.join(coef_self)
plot_heatmap(
    coef_combined,
    title="Learned Coef weights for final parent and self models",
    vmin=-2.5,
    vmax=2.5,
    xlabel="Cohort",
)

# Saliency Sanity Checks

In [ ]:
def get_saliency_map(config_dict, model, df, coef_df):
    # Load necessary vars
    impact_type = config_dict["impact_type"]
    quest_pref = "p_dep" if impact_type == "parent" else "s_dep"
    # Expand text prob vector into cols
    df_sub = model.expand_text(df)
    # Add sum score col
    quest_cols = model.quest_cols
    df_sub["sum_score"] = (
        df_sub[quest_cols].to_numpy(dtype=int).sum(axis=1, keepdims=True)
    )
    ## Multiply learned coefs w/ observed values
    new_cols = []
    for idx, row in coef_df.iterrows():
        col_name = "_".join(idx.split("_")[1:])  # remove stage_ prefix
        # stage = idx_split[0]
        if col_name == "intercept":
            continue
        if col_name.startswith("_"):  # add prefix to quest cols
            col_name = quest_pref + col_name
        # Multiply coef scalar by observed vals for current feature
        coef = getattr(row, f"coef_{impact_type}")
        df_col = df_sub[col_name]
        df_col_weighted = df_col * coef
        new_cols.append(pd.DataFrame({col_name: df_col_weighted}))
    # Create combined saliency map
    combined = pd.concat(new_cols, axis=1)
    # Summarize across all participants
    summary = combined.describe().loc[["mean", "std", "min", "max"]]
    # plot
    plot_heatmap(
        summary.T,
        title="Summary of saliency",
        vmin=-0.5,
        vmax=0.5,
        xlabel="Summary type",
    )
    return summary

In [ ]:
saliency_self = get_saliency_map(self_config, model_self, df_self, coef_self)

In [ ]:
saliency_parent = get_saliency_map(parent_config, model_parent, df_parent, coef_parent)

# LMG Feat Imp

**TODO:**

- Randomly sample order of text outcomes 1000 times
- Iteratively compute incremental r^2 after adding each outcome
- Avg across all 1000 samples to get avg contribution to accuracy